In [1]:
import pandas as pd
import numpy as np
from nba_api.stats.endpoints import leaguedashplayerstats
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.stats.outliers_influence import variance_inflation_factor
from scipy import stats
import warnings
from nba_api.stats.endpoints import leaguedashplayerstats
import time
from IPython.display import display

Cleaning Data Gang 

In [2]:
salaries = pd.read_csv("NBA Player Salaries.csv") 
salaries = salaries.drop(columns=["Player Id"])
salaries = salaries.rename( columns = {'2024/2025.1' : '2025/2026'}) 
salaries.head() 


,Player Name,2022/2023,2023/2024,2024/2025,2025/2026
0,Stephen Curry,"$48,070,014","$51,915,615","$55,761,217","$59,606,817"
1,John Wall,"$47,345,760",$0,$0,$0
2,Russell Westbrook,"$47,080,179",$0,$0,$0
3,LeBron James,"$44,474,988","$46,698,737","$50,434,636",$0
4,Kevin Durant,"$44,119,845","$47,649,433","$51,179,020","$54,708,608"


In [3]:
stats = leaguedashplayerstats.LeagueDashPlayerStats(
    season="2024-25",
    season_type_all_star="Regular Season"
)

df = stats.get_data_frames()[0]

df.head()

,PLAYER_ID,PLAYER_NAME,NICKNAME,TEAM_ID,TEAM_ABBREVIATION,AGE,GP,W,L,W_PCT,...,BLKA_RANK,PF_RANK,PFD_RANK,PTS_RANK,PLUS_MINUS_RANK,NBA_FANTASY_PTS_RANK,DD2_RANK,TD3_RANK,WNBA_FANTASY_PTS_RANK,TEAM_COUNT
0,1630639,A.J. Lawson,A.J.,1610612761,TOR,24.0,26,14,12,0.538,...,241,200,327,337,299,375,159,44,364,1
1,1631260,AJ Green,AJ,1610612749,MIL,25.0,73,44,29,0.603,...,99,498,290,214,62,247,281,44,225,1
2,1642358,AJ Johnson,AJ,1610612764,WAS,20.0,29,8,21,0.276,...,312,217,369,347,495,379,281,44,372,2
3,203932,Aaron Gordon,Aaron,1610612743,DEN,29.0,51,33,18,0.647,...,451,301,107,139,41,183,134,44,181,1
4,1628988,Aaron Holiday,Aaron,1610612745,HOU,28.0,62,39,23,0.629,...,224,259,292,292,115,322,281,44,315,1


In [4]:


import pandas as pd
import time
from nba_api.stats.endpoints import leaguedashplayerstats
from IPython.display import display

seasons = ["2022-23", "2023-24", "2024-25", "2025-26"]

all_stats = []

for season in seasons:
    stats = leaguedashplayerstats.LeagueDashPlayerStats(
        season=season,
        season_type_all_star="Regular Season",
        per_mode_detailed="PerGame"
    )

    df = stats.get_data_frames()[0]
    df["SEASON"] = season
    df["SEASON_TYPE"] = "Regular Season"

    all_stats.append(df)

    time.sleep(1)

nba_stats = pd.concat(all_stats, ignore_index=True)

basic_stats = [
    "PLAYER_NAME",
    "SEASON",
    "SEASON_TYPE",
    "TEAM_ABBREVIATION",
    "GP",
    "MIN",
    "PTS",
    "FGM",
    "FG3M",
    "FTM",
    "REB",
    "AST",
    "STL",
    "BLK",
    "TOV",
    "PF",
    "PLUS_MINUS"
]

nba_basic_stats = nba_stats[basic_stats]

display(nba_basic_stats)



,PLAYER_NAME,SEASON,SEASON_TYPE,TEAM_ABBREVIATION,GP,MIN,PTS,FGM,FG3M,FTM,REB,AST,STL,BLK,TOV,PF,PLUS_MINUS
0,A.J. Lawson,2022-23,Regular Season,DAL,15,7.2,3.7,1.5,0.7,0.1,1.4,0.1,0.1,0.0,0.2,0.7,-3.1
1,AJ Green,2022-23,Regular Season,MIL,35,9.9,4.4,1.5,1.3,0.1,1.3,0.6,0.2,0.0,0.3,0.9,-0.7
2,AJ Griffin,2022-23,Regular Season,ATL,72,19.5,8.9,3.4,1.4,0.6,2.1,1.0,0.6,0.2,0.6,1.2,0.9
3,Aaron Gordon,2022-23,Regular Season,DEN,68,30.2,16.3,6.3,0.9,2.8,6.6,3.0,0.8,0.8,1.4,1.9,7.6
4,Aaron Holiday,2022-23,Regular Season,ATL,63,13.4,3.9,1.5,0.6,0.4,1.2,1.4,0.6,0.2,0.6,1.3,0.3
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2257,Zach LaVine,2025-26,Regular Season,SAC,39,31.4,19.2,6.7,2.5,3.2,2.8,2.3,0.7,0.3,1.9,2.1,-7.9
2258,Zeke Nnaji,2025-26,Regular Season,DEN,52,12.0,3.7,1.4,0.3,0.7,2.6,0.6,0.3,0.5,0.5,1.4,-2.1
2259,Ziaire Williams,2025-26,Regular Season,BKN,56,22.9,10.2,3.3,1.5,2.1,2.4,1.1,1.4,0.4,1.1,2.1,-5.0
2260,Zion Williamson,2025-26,Regular Season,NOP,62,29.7,21.0,7.8,0.0,5.4,5.7,3.2,1.0,0.5,2.0,2.3,-1.3


In [5]:
salaries_long = salaries.melt(
    id_vars="Player Name",
    var_name="SEASON",
    value_name="Salary"
)
salaries_long.head() 



,Player Name,SEASON,Salary
0,Stephen Curry,2022/2023,"$48,070,014"
1,John Wall,2022/2023,"$47,345,760"
2,Russell Westbrook,2022/2023,"$47,080,179"
3,LeBron James,2022/2023,"$44,474,988"
4,Kevin Durant,2022/2023,"$44,119,845"


In [6]:
salaries_long["Salary"] = (
    salaries_long["Salary"]
    .astype(str)
    .str.replace("$", "", regex=False)
    .str.replace(",", "", regex=False)
)

salaries_long["Salary"] = pd.to_numeric(salaries_long["Salary"], errors="coerce")

salaries_long = salaries_long[salaries_long["Salary"] != 0]

salaries_long = salaries_long.reset_index(drop=True)

salaries_long.head()

,Player Name,SEASON,Salary
0,Stephen Curry,2022/2023,48070014
1,John Wall,2022/2023,47345760
2,Russell Westbrook,2022/2023,47080179
3,LeBron James,2022/2023,44474988
4,Kevin Durant,2022/2023,44119845


In [7]:
# Rename salary player column to match NBA API column
salaries_long = salaries_long.rename(columns={
    "Player Name": "PLAYER_NAME"
})

nba_basic_stats["PLAYER_NAME"] = nba_basic_stats["PLAYER_NAME"].str.strip().str.lower()
salaries_long["PLAYER_NAME"] = salaries_long["PLAYER_NAME"].str.strip().str.lower()

nba_basic_stats["SEASON"] = nba_basic_stats["SEASON"].astype(str).str.strip()
salaries_long["SEASON"] = salaries_long["SEASON"].astype(str).str.strip()

salaries_long["SEASON"] = salaries_long["SEASON"].str.replace(".1", "", regex=False)

salaries_long["SEASON"] = (
    salaries_long["SEASON"].str[:4] + "-" + salaries_long["SEASON"].str[-2:]
)
merged = nba_basic_stats.merge(
    salaries_long,
    on=["PLAYER_NAME", "SEASON"],
    how="inner"
)

merged.head()

,PLAYER_NAME,SEASON,SEASON_TYPE,TEAM_ABBREVIATION,GP,MIN,PTS,FGM,FG3M,FTM,REB,AST,STL,BLK,TOV,PF,PLUS_MINUS,Salary
0,aj green,2022-23,Regular Season,MIL,35,9.9,4.4,1.5,1.3,0.1,1.3,0.6,0.2,0.0,0.3,0.9,-0.7,508891
1,aj griffin,2022-23,Regular Season,ATL,72,19.5,8.9,3.4,1.4,0.6,2.1,1.0,0.6,0.2,0.6,1.2,0.9,3536160
2,aaron gordon,2022-23,Regular Season,DEN,68,30.2,16.3,6.3,0.9,2.8,6.6,3.0,0.8,0.8,1.4,1.9,7.6,19690909
3,aaron holiday,2022-23,Regular Season,ATL,63,13.4,3.9,1.5,0.6,0.4,1.2,1.4,0.6,0.2,0.6,1.3,0.3,1968175
4,aaron nesmith,2022-23,Regular Season,IND,73,24.9,10.1,3.5,1.6,1.6,3.8,1.3,0.8,0.5,1.0,3.2,-2.9,3804360


In [12]:
merged_nameA_TO_Z = merged.sort_values("PLAYER_NAME").reset_index(drop=True)

display(merged)

,PLAYER_NAME,SEASON,SEASON_TYPE,TEAM_ABBREVIATION,GP,MIN,PTS,FGM,FG3M,FTM,REB,AST,STL,BLK,TOV,PF,PLUS_MINUS,Salary
0,aj green,2022-23,Regular Season,MIL,35,9.9,4.4,1.5,1.3,0.1,1.3,0.6,0.2,0.0,0.3,0.9,-0.7,508891
1,aj griffin,2022-23,Regular Season,ATL,72,19.5,8.9,3.4,1.4,0.6,2.1,1.0,0.6,0.2,0.6,1.2,0.9,3536160
2,aaron gordon,2022-23,Regular Season,DEN,68,30.2,16.3,6.3,0.9,2.8,6.6,3.0,0.8,0.8,1.4,1.9,7.6,19690909
3,aaron holiday,2022-23,Regular Season,ATL,63,13.4,3.9,1.5,0.6,0.4,1.2,1.4,0.6,0.2,0.6,1.3,0.3,1968175
4,aaron nesmith,2022-23,Regular Season,IND,73,24.9,10.1,3.5,1.6,1.6,3.8,1.3,0.8,0.5,1.0,3.2,-2.9,3804360
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1061,tyler herro,2025-26,Regular Season,MIA,33,31.3,20.5,7.5,2.5,3.0,4.8,4.1,0.7,0.4,1.9,1.9,-1.2,31000000
1062,walker kessler,2025-26,Regular Season,UTA,5,30.8,14.4,5.2,1.2,2.8,10.8,3.0,1.4,1.8,3.2,4.4,2.8,4878938
1063,zach lavine,2025-26,Regular Season,SAC,39,31.4,19.2,6.7,2.5,3.2,2.8,2.3,0.7,0.3,1.9,2.1,-7.9,45999660
1064,ziaire williams,2025-26,Regular Season,BKN,56,22.9,10.2,3.3,1.5,2.1,2.4,1.1,1.4,0.4,1.1,2.1,-5.0,8353152
